In [0]:
from pyspark.sql.functions import col, to_date, trim, upper, lower, when, current_timestamp


In [0]:
bronze_orders = spark.table("ecommerce_project_1.ecommerce_project_1.bronze_orders")

silver_orders = (
    bronze_orders
    # Type casting
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("price", col("price").cast("double"))
    
    # Trim & normalize string columns
    .withColumn("status", trim(upper(col("status"))))    # e.g., DELIVERED, SHIPPED
    .withColumn("country", trim(upper(col("country"))))  # INDIA, USA, etc.
    
    # Filter out bad data
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("price") > 0)
    
    # Add/update ingestion/update timestamp if needed
    .withColumn("updated_ts", current_timestamp())
    
    # Remove duplicates
    .dropDuplicates(["order_id", "product_id"])
)

(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project_1.ecommerce_project_1.silver_orders")
)


In [0]:
display(spark.read.table("ecommerce_project_1.ecommerce_project_1.silver_orders").limit(5))

order_id,order_date,customer_id,product_id,quantity,price,status,country,ingest_ts,updated_ts
249,2024-01-03,37,1017,3,621.03,CANCELLED,AUSTRALIA,2025-11-25T06:36:05.195Z,2025-11-25T07:20:00.929Z
194,2024-02-03,24,1013,3,1145.38,RETURNED,AUSTRALIA,2025-11-25T06:36:05.195Z,2025-11-25T07:20:00.929Z
281,2024-01-12,25,1017,2,1554.67,DELIVERED,UK,2025-11-25T06:36:05.195Z,2025-11-25T07:20:00.929Z
252,2024-02-01,40,1007,4,596.08,CANCELLED,CANADA,2025-11-25T06:36:05.195Z,2025-11-25T07:20:00.929Z
400,2024-02-20,5,1030,2,1693.37,RETURNED,GERMANY,2025-11-25T06:36:05.195Z,2025-11-25T07:20:00.929Z


In [0]:
bronze_customers = spark.table("ecommerce_project_1.ecommerce_project_1.bronze_customers")

silver_customers = (
    bronze_customers
    .withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd"))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("city", trim(col("city")))
    .withColumn("country", upper(trim(col("country"))))
    .filter(col("customer_id").isNotNull())
    .filter(col("email").isNotNull())
    .dropDuplicates(["customer_id"])
    .withColumn("updated_ts", current_timestamp())
)

(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project_1.ecommerce_project_1.silver_customers")
)


In [0]:
display(spark.read.table("ecommerce_project_1.ecommerce_project_1.silver_customers").limit(5))

customer_id,name,email,signup_date,city,country,ingest_ts,updated_ts
29,Customer_29,customer29@mail.com,2023-01-29,Delhi,GERMANY,2025-11-25T06:39:40.329Z,2025-11-25T07:22:20.994Z
24,Customer_24,customer24@mail.com,2023-09-30,Toronto,INDIA,2025-11-25T06:39:40.329Z,2025-11-25T07:22:20.994Z
39,Customer_39,customer39@mail.com,2023-09-16,Delhi,INDIA,2025-11-25T06:39:40.329Z,2025-11-25T07:22:20.994Z
38,Customer_38,customer38@mail.com,2023-08-10,Berlin,USA,2025-11-25T06:39:40.329Z,2025-11-25T07:22:20.994Z
15,Customer_15,customer15@mail.com,2022-12-31,Berlin,AUSTRALIA,2025-11-25T06:39:40.329Z,2025-11-25T07:22:20.994Z


In [0]:
bronze_products = spark.table("ecommerce_project_1.ecommerce_project_1.bronze_products")

silver_products = (
    bronze_products
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(upper(col("category"))))
    .withColumn("sub_category", trim(upper(col("sub_category"))))
    .filter(col("product_id").isNotNull())
    .filter(col("unit_price") > 0)
    .dropDuplicates(["product_id"])
    .withColumn("updated_ts", current_timestamp())
)

(
    silver_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project_1.ecommerce_project_1.silver_products")
)


In [0]:
display(spark.read.table("ecommerce_project_1.ecommerce_project_1.silver_products").limit(5))

product_id,product_name,category,sub_category,unit_price,ingest_ts,updated_ts
1021,Product_1021,ACCESSORIES,STANDARD,1189.22,2025-11-25T06:39:43.166Z,2025-11-25T07:23:50.488Z
1007,Product_1007,HOME,PREMIUM,513.13,2025-11-25T06:39:43.166Z,2025-11-25T07:23:50.488Z
1006,Product_1006,ACCESSORIES,STANDARD,1542.24,2025-11-25T06:39:43.166Z,2025-11-25T07:23:50.488Z
1015,Product_1015,WEARABLE,PREMIUM,750.69,2025-11-25T06:39:43.166Z,2025-11-25T07:23:50.488Z
1009,Product_1009,ACCESSORIES,STANDARD,691.38,2025-11-25T06:39:43.166Z,2025-11-25T07:23:50.488Z
